# Tema 1 — Lectura y manipulación de datos con pandas

| Recurso | Fichero |
| --- | --- |
| **Ejercicio pandas (E1)** | [`5_ejercicios_pandas.md`](5_ejercicios_pandas.md) |
| Ejercicios (datos + ML) | [`7_ejercicios_datos_y_modelos.md`](7_ejercicios_datos_y_modelos.md) |
| Datasets E1 pandas | [`Datos/ventas.csv`](Datos/ventas.csv) · [`Datos/penguins.csv`](Datos/penguins.csv) |
| Orígenes de datos | [`Datos/README.md`](Datos/README.md) |
| Dataset auxiliar | [`Datos/iris.csv`](Datos/iris.csv) |
| Mismos datos iris (JSON / Excel) | [`Datos/iris.json`](Datos/iris.json) · [`Datos/iris.xlsx`](Datos/iris.xlsx) |
| Documentación | [pandas.pydata.org/docs](https://pandas.pydata.org/docs/) |

Este notebook es la **referencia amplia** de la sesión de lectura de datos en DAVD: conceptos, demos ejecutables, flujos recomendados, anti-patrones y puente a modelos y visualizaciones. Úsalo en clase y como manual cuando prepares datos para un cuadro de mando.

**Cómo trabajarlo:** ejecuta las celdas en orden con el entorno del curso activado (`pandas` instalado; para Excel suele hacer falta `openpyxl`).


---

## 0. Objetivos de aprendizaje

Al terminar esta sesión (y este notebook) deberías ser capaz de:

1. Explicar por qué pandas es la base de **lectura y transformación** antes de visualizar o modelar.
2. Distinguir `Series` y `DataFrame` y crearlos de forma controlada.
3. Cargar **CSV, JSON y Excel** con **`pathlib`** (sin rutas absolutas del autor).
4. Diagnosticar un dataset: `shape`, `dtypes`, nulos, distribuciones.
5. Filtrar, seleccionar (`loc`/`iloc`), agrupar y agregar.
6. Crear columnas derivadas con operaciones vectoriales (evitar `apply` innecesario).
7. Exportar resultados listos para la siguiente etapa (modelo o dashboard).
8. Enlazar este flujo con los ejercicios de lectura multi-formato y con scikit-learn.
9. Completar el **E1 pandas** (`5_ejercicios_pandas.md`): pipeline sobre `ventas.csv` **y** el dataset abierto `penguins.csv`.

---


## 1. Por qué esta sesión importa en DAVD

### 1.1 Qué entregamos

En DAVD no entregamos “un Excel retocado a mano”. Entregamos **aplicaciones y cuadros de mando** que consumen datos de forma reproducible:

```text
datos crudos → lectura → diagnóstico → transformación → (modelo) → visualización / Dash → despliegue
```

Si la lectura y la limpieza viven solo en celdas frágiles, el dashboard se rompe en cuanto cambias de máquina o de fichero.

### 1.2 Mapa mental del semestre

```text
Tema 1  →  Python, pandas (hoy), modelos, viz básicas
Tema 2  →  Plotly / Dash / callbacks
Tema 3  →  entornos, APIs, despliegue
```

Todo lo posterior **asume** que sabes cargar datos de forma portable y dejar un DataFrame limpio.

### 1.3 Analogías útiles

| Concepto | Analogía |
| --- | --- |
| `Series` | Una columna etiquetada |
| `DataFrame` | Tabla (colecciones de Series alineadas) |
| `dtype` | El “tipo de casilla” de la hoja |
| filtro booleano | Quedarse con las filas que cumplen una regla |
| `groupby` | Tablas dinámicas de Excel, pero en código |
| `pathlib` | GPS relativo al proyecto, no “en mi Escritorio” |
| exportar CSV/JSON | Entregar el dataset listo al siguiente módulo (modelo / Dash) |

### 1.4 Regla de oro

> **Explora en el notebook; deja rutas portables; exporta un artefacto que otra persona pueda volver a generar.**

---


## 2. Entorno e imports

Comprueba que el intérprete del notebook es el del entorno del curso.


In [1]:
from __future__ import annotations

from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

print("pandas:", pd.__version__)
print("cwd:", Path.cwd())
DATA_DIR = Path("Datos")  # ejecuta desde 1_Introduccion_Python/
print("Datos existe:", DATA_DIR.exists())
print("ficheros:", sorted(p.name for p in DATA_DIR.glob("iris.*")))


pandas: 1.4.3
cwd: /Users/lucia/Desktop/DAVD/davd-26-27/1_Introduccion_Python
Datos existe: True
ficheros: ['iris.csv', 'iris.json', 'iris.xlsx']


Si `Datos existe: False`, cambia el directorio de trabajo del notebook a `1_Introduccion_Python/` (o ajusta `DATA_DIR`).

Para Excel:

```bash
python -m pip install openpyxl
```

---

## 3. `Series`: la columna tipada

Una **Series** es una estructura **unidimensional** con índice. Documentación: [`pandas.Series`](https://pandas.pydata.org/docs/reference/api/pandas.Series.html).

### 3.1 Creación básica


In [2]:
serie = pd.Series(["a", "b", "c"])
serie


0    a
1    b
2    c
dtype: object

### 3.2 Series con índice semántico (p. ej. fechas)


In [3]:
ts = pd.Series(
    {
        date(2026, 9, 14): 10,
        date(2026, 9, 21): 15,
        date(2026, 9, 28): 15,
    }
)
ts


2026-09-14    10
2026-09-21    15
2026-09-28    15
dtype: int64

**Para DAVD:** casi siempre trabajarás con columnas dentro de un `DataFrame` que luego alimentarás a Plotly/Dash o a un modelo.

---

## 4. `DataFrame`: la tabla de trabajo

Un **DataFrame** es una estructura **bidimensional** (filas × columnas). Cada columna es una Series. Documentación: [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).

### 4.1 Crear desde filas + nombres de columnas


In [4]:
df_demo = pd.DataFrame(
    data=[
        [date(2026, 9, 14), 1, "a"],
        [date(2026, 9, 21), 2, "b"],
        [date(2026, 9, 28), 3, "c"],
    ],
    columns=["timestamp", "integers", "strings"],
)
df_demo


,timestamp,integers,strings
0,2026-09-14,1,a
1,2026-09-21,2,b
2,2026-09-28,3,c


### 4.2 Índice: cuidado con `inplace`

`set_index` **devuelve una copia** salvo que uses `inplace=True` (preferible asignar).


In [5]:
df_indexed = df_demo.set_index("timestamp")
df_indexed


,integers,strings
timestamp,,
2026-09-14,1,a
2026-09-21,2,b
2026-09-28,3,c


### 4.3 Crear columna a columna


In [6]:
df2 = pd.DataFrame()
df2["timestamp"] = [date(2026, 9, 14), date(2026, 9, 21), date(2026, 9, 28)]
df2["integers"] = [1, 2, 3]
df2["strings"] = ["a", "b", "c"]
df2


,timestamp,integers,strings
0,2026-09-14,1,a
1,2026-09-21,2,b
2,2026-09-28,3,c


In [7]:
# Comparación elemento a elemento
df_demo.reset_index(drop=True) == df2


,timestamp,integers,strings
0,True,True,True
1,True,True,True
2,True,True,True


### 4.4 Paso a NumPy

Útil cuando pasas features a scikit-learn. Preferible `to_numpy()` frente a `.values`.


In [8]:
df_demo.to_numpy()


array([[datetime.date(2026, 9, 14), 1, 'a'],
       [datetime.date(2026, 9, 21), 2, 'b'],
       [datetime.date(2026, 9, 28), 3, 'c']], dtype=object)

---

## 5. Lectura de datos (I/O en el borde)

### 5.1 Rutas: absoluta vs relativa

| Tipo | Ejemplo | Problema |
| --- | --- | --- |
| Absoluta | `/Users/yo/Desktop/iris.csv` | No funciona en otra máquina |
| Relativa al tema | `Datos/iris.csv` | Portable si fijas el cwd |

En DAVD usamos **`pathlib.Path`** relativo al proyecto.

### 5.2 CSV

Documentación: [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).


In [9]:
iris_csv = DATA_DIR / "iris.csv"
iris = pd.read_csv(iris_csv)
iris.head()


,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa


### 5.3 JSON

Documentación: [`read_json`](https://pandas.pydata.org/docs/reference/api/pandas.read_json.html).


In [10]:
iris_json = pd.read_json(DATA_DIR / "iris.json")
iris_json.head()


,sepalLength,sepalWidth,petalLength,petalWidth,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


### 5.4 Excel

Documentación: [`read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html).


In [ ]:
from pathlib import Path
import zipfile

archivo = DATA_DIR / "iris.xlsx"

print("Ruta:", archivo.resolve())
print("Existe:", archivo.exists())
print("Tamaño:", archivo.stat().st_size, "bytes")
print("Es un Excel válido:", zipfile.is_zipfile(archivo))

Ruta: /Users/lucia/Desktop/DAVD/davd-26-27/1_Introduccion_Python/Datos/iris.xlsx
Existe: True
Tamaño: 27648 bytes
Es un Excel válido: False


In [14]:
with open(DATA_DIR / "iris.xlsx", "rb") as f:
    print(f.read(20))

b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'


In [15]:
%pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [16]:
iris_xlsx = pd.read_excel(DATA_DIR / "iris.xlsx", engine="xlrd")
iris_xlsx.head()


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NaN,Sepal Length (cm),Sepal Width (cm),Petal Length (cm),Petal Width (cm),Class,NaN,NaN,alpha,obj,NaN,NaN
1,NaN,7,3.2,4.7,1.4,Iris-versicolor,NaN,0.0,0,0,0.0,1.0
2,NaN,6.4,3.2,4.5,1.5,Iris-versicolor,NaN,0.0,NaN,NaN,0.0,1.0
3,NaN,6.9,3.1,4.9,1.5,Iris-versicolor,NaN,0.0,NaN,NaN,0.0,1.0
4,NaN,5.5,2.3,4,1.3,Iris-versicolor,NaN,0.0,NaN,NaN,0.0,1.0


### 5.5 ¿Son el mismo dataset?

En un dashboard o un modelo, conviene comprobar que el origen no cambia el contenido.


In [17]:
# Normalizamos nombres de columnas para comparar
def normalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out.columns = [str(c).replace(".", "_") for c in out.columns]
    return out.reset_index(drop=True)


a = normalize_columns(iris)
b = normalize_columns(iris_json)
c = normalize_columns(iris_xlsx)
print("CSV vs JSON iguales:", a.equals(b))
print("CSV vs Excel iguales:", a.equals(c))
print("shape:", a.shape)


CSV vs JSON iguales: False
CSV vs Excel iguales: False
shape: (150, 5)


### 5.6 Función única de carga (adelanto del ejercicio E1)

Idea que reutilizarás con `airports` u otros ficheros:


In [18]:
def leer_tabla(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No existe el fichero: {path}")
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, engine="openpyxl")
    raise ValueError(f"Formato no soportado: {suffix}")


leer_tabla(DATA_DIR / "iris.csv").head(2)


,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa


---

## 6. Diagnóstico de un DataFrame (antes de visualizar o modelar)

Orden recomendado:

```text
cargar → shape/dtypes → head → na → describe/value_counts → sospechosos
```

Trabajamos ya con nombres de columna normalizados.


In [19]:
iris = normalize_columns(pd.read_csv(DATA_DIR / "iris.csv"))
print("shape:", iris.shape)
print()
print("dtypes:")
print(iris.dtypes)
print()
print("nulos por columna:")
print(iris.isna().sum())


shape: (150, 5)

dtypes:
sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width     float64
variety          object
dtype: object

nulos por columna:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
variety         0
dtype: int64


In [20]:
iris.head()


,sepal_length,sepal_width,petal_length,petal_width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa


In [21]:
iris.describe(include="all")


,sepal_length,sepal_width,petal_length,petal_width,variety
count,150.000000,150.000000,150.000000,150.000000,150
unique,NaN,NaN,NaN,NaN,3
top,NaN,NaN,NaN,NaN,Setosa
freq,NaN,NaN,NaN,NaN,50
mean,5.843333,3.057333,3.758000,1.199333,NaN
std,0.828066,0.435866,1.765298,0.762238,NaN
min,4.300000,2.000000,1.000000,0.100000,NaN
25%,5.100000,2.800000,1.600000,0.300000,NaN
50%,5.800000,3.000000,4.350000,1.300000,NaN
75%,6.400000,3.300000,5.100000,1.800000,NaN


In [22]:
iris["variety"].value_counts()


Setosa        50
Versicolor    50
Virginica     50
Name: variety, dtype: int64

**Lectura crítica:** ¿hay desbalance entre clases? ¿escalas muy distintas entre `sepal_*` y `petal_*`? Eso importa después en ML y en ejes de gráficos.

---

## 7. Selección, filtros y localización

### 7.1 Filtro booleano


In [23]:
iris[iris["sepal_length"] >= iris["sepal_length"].mean()].head()


,sepal_length,sepal_width,petal_length,petal_width,variety
50,7.0,3.2,4.7,1.4,Versicolor
51,6.4,3.2,4.5,1.5,Versicolor
52,6.9,3.1,4.9,1.5,Versicolor
54,6.5,2.8,4.6,1.5,Versicolor
56,6.3,3.3,4.7,1.6,Versicolor


### 7.2 `loc` (etiquetas) vs `iloc` (posición)


In [24]:
iris.loc[0, "sepal_length"]


5.1

In [25]:
iris.iloc[19]


sepal_length       5.1
sepal_width        3.8
petal_length       1.5
petal_width        0.3
variety         Setosa
Name: 19, dtype: object

**Heurística:** usa nombres de columna en pipelines y callbacks; `iloc` solo cuando la posición importa de verdad.

---

## 8. Agrupaciones y agregaciones

Útiles para KPIs del dashboard (medias por categoría, máximos, conteos).

### 8.1 Media por categoría


In [26]:
iris.groupby("variety")["sepal_length"].mean()


variety
Setosa        5.006
Versicolor    5.936
Virginica     6.588
Name: sepal_length, dtype: float64

### 8.2 Varias métricas con `agg`


In [27]:
iris.groupby("variety").agg(
    petal_length_median=("petal_length", "median"),
    petal_width_max=("petal_width", "max"),
    n=("variety", "size"),
)


,petal_length_median,petal_width_max,n
variety,,,
Setosa,1.50,0.6,50
Versicolor,4.35,1.8,50
Virginica,5.55,2.5,50


---

## 9. Operaciones vectoriales (preferibles a bucles)

### 9.1 Aritmética columna a columna


In [28]:
(iris["sepal_length"] / iris["petal_length"]).head()


0    3.642857
1    3.500000
2    3.615385
3    3.066667
4    3.571429
dtype: float64

### 9.2 `map` / `apply`: úsalos con criterio

- **Vectorizado** → rápido y claro.
- `Series.map` → transformaciones elemento a elemento simples.
- `DataFrame.apply(..., axis=1)` → flexible pero lento; último recurso.


In [29]:
iris["petal_width"].map(lambda x: int(x) if x > 1.5 else x).head()


0    0.2
1    0.2
2    0.2
3    0.2
4    0.2
Name: petal_width, dtype: float64

In [30]:
# apply por filas (pedagógico)
iris.apply(lambda row: 5 * row["petal_length"] + 6 * row["petal_width"], axis=1).head()


0    8.2
1    8.2
2    7.7
3    8.7
4    8.2
dtype: float64

Equivalente vectorizado (preferible):


In [31]:
(5 * iris["petal_length"] + 6 * iris["petal_width"]).head()


0    8.2
1    8.2
2    7.7
3    8.7
4    8.2
dtype: float64

---

## 10. Limpieza básica: tipos, duplicados, columnas

### 10.1 Cambiar tipo


In [32]:
iris["sepal_width"].astype(str).iloc[0]


'3.5'

### 10.2 Duplicados y borrado de columnas (sin romper el original)


In [33]:
print("duplicados:", int(iris.duplicated().sum()))
iris_no_dupes = iris.drop_duplicates()
iris_without_width = iris.drop(columns=["sepal_width"])
iris_without_width.head()


duplicados: 1


,sepal_length,petal_length,petal_width,variety
0,5.1,1.4,0.2,Setosa
1,4.9,1.4,0.2,Setosa
2,4.7,1.3,0.2,Setosa
3,4.6,1.5,0.2,Setosa
4,5.0,1.4,0.2,Setosa


**Hábitos:**

- Preferir `drop(columns=[...])`.
- Evitar `inplace=True` en material de curso.
- No mutar el DataFrame “crudo” si luego quieres comparar orígenes.

---

## 11. Mini-pipeline DAVD: de fichero a artefacto listo

Patrón que reutilizarás hacia modelos y Dash:

```text
cargar → normalizar → diagnosticar → transformar → exportar
```

### 11.1 Preparar features y target (puente a scikit-learn)


In [34]:
def preparar_iris(path: Path) -> tuple[pd.DataFrame, pd.Series]:
    frame = normalize_columns(leer_tabla(path))
    features = frame.drop(columns=["variety"])
    target = frame["variety"]
    return features, target


X, y = preparar_iris(DATA_DIR / "iris.csv")
print("X:", X.shape, "| y clases:", y.value_counts().to_dict())
X.head()


X: (150, 4) | y clases: {'Setosa': 50, 'Versicolor': 50, 'Virginica': 50}


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


### 11.2 Resumen para un cuadro de mando


In [35]:
resumen = (
    iris.groupby("variety", as_index=False)
    .agg(
        sepal_length_mean=("sepal_length", "mean"),
        petal_length_mean=("petal_length", "mean"),
        n=("variety", "size"),
    )
    .sort_values("n", ascending=False)
)
resumen


,variety,sepal_length_mean,petal_length_mean,n
0,Setosa,5.006,1.462,50
1,Versicolor,5.936,4.260,50
2,Virginica,6.588,5.552,50


### 11.3 Exportar artefactos


In [36]:
salida_limpia = DATA_DIR / "iris_limpia.csv"
salida_resumen = DATA_DIR / "iris_resumen.csv"

iris.to_csv(salida_limpia, index=False)
resumen.to_csv(salida_resumen, index=False)
print("escrito:", salida_limpia)
print("escrito:", salida_resumen)


escrito: Datos/iris_limpia.csv
escrito: Datos/iris_resumen.csv


> **Nota:** los CSV generados son salidas. Puedes no versionarlos; sí debes saber regenerarlos.

---

## 12. Anti-patrones frecuentes (lista negra)

1. Rutas absolutas (`/Users/.../iris.csv`).
2. Tres celdas distintas para CSV/JSON/Excel sin una función común.
3. Sobrescribir el DataFrame crudo sin `copy()`.
4. `for row in df.iterrows()` para todo.
5. `apply(axis=1)` cuando basta una operación vectorial.
6. Visualizar o entrenar **antes** de mirar nulos y dtypes.
7. Encadenar 30 transformaciones en una sola celda sin nombres.
8. Depender del orden mágico de “Run All”.
9. Dejar el resultado solo en pantalla: sin CSV/JSON no hay pipeline ni Dash.
10. Mezclar lectura, modelo y layout de Dash en un único bloque ilegible.

---

## 13. De notebook a aplicación (adelanto Tema 2–3)

Hoy exploramos aquí. Más adelante querrás:

```text
datos/          ← CSV/JSON o descarga
src/data.py     ← leer_tabla / preparar_*
src/model.py    ← entrenamiento (notebook 6)
app.py          ← Dash que lee el DataFrame limpio
```

Misma lógica; mejor frontera.

---

## 14. Autoevaluación

1. ¿Qué diferencia una Series de un DataFrame?
2. ¿Por qué `pathlib` frente a rutas absolutas?
3. ¿Cómo cargarías CSV, JSON o Excel con una sola función?
4. ¿Qué imprime tu diagnóstico antes de plotear o entrenar?
5. ¿Cuándo evitarías `apply(axis=1)`?
6. ¿Qué exportarías para alimentar un callback de Dash?

Si dudas en más de dos, rehaz las secciones 5–11 y el ejercicio E1 de [`7_ejercicios_datos_y_modelos.md`](7_ejercicios_datos_y_modelos.md).

---

## 15. Checklist de salida

### Código

- [ ] Notebook ejecuta desde `1_Introduccion_Python/`
- [ ] `iris.csv` / `.json` / `.xlsx` cargados con `Path`
- [ ] Diagnóstico: `shape`, `dtypes`, nulos, `value_counts`
- [ ] Filtros, `groupby` y operación vectorizada probados
- [ ] Función `leer_tabla` (o equivalente) escrita
- [ ] Exportados `iris_limpia.csv` y `iris_resumen.csv`
- [ ] E1 pandas: `validar_ventas` → 140/10 y JSON de calidad

### Comprensión

- [ ] Sé explicar el flujo cargar → transformar → exportar → viz/modelo
- [ ] Sé 3 anti-patrones que no cometeré en el proyecto
- [ ] Sé qué llevaré al notebook de modelos y a Dash

---

## 16. Para la siguiente sesión

1. Completa el **E1 pandas** en [`5_ejercicios_pandas.md`](5_ejercicios_pandas.md) (ventas: 140 válidas / 10 inválidas).
2. Completa el E1 de airports multi-formato en [`7_ejercicios_datos_y_modelos.md`](7_ejercicios_datos_y_modelos.md).
3. Ojea `6_modelos_machine_learning.ipynb`: ahí `X` / `y` ya no serán un misterio.
4. Piensa qué columnas de **tu proyecto** necesitarás leer y limpiar para el dashboard.

---

## 17. Apéndice A — Chuleta rápida

```python
from pathlib import Path
import pandas as pd

DATA = Path("Datos")
df = pd.read_csv(DATA / "iris.csv")
df = df.rename(columns=lambda c: str(c).replace(".", "_"))
df.shape, df.dtypes, df.isna().sum()
df.head(); df.describe(include="all")
df[df["variety"] == "Setosa"]
df.loc[0, "sepal_length"]; df.iloc[0]
df.groupby("variety")["sepal_length"].mean()
df.to_csv(DATA / "salida.csv", index=False)
pd.read_json(DATA / "iris.json")
pd.read_excel(DATA / "iris.xlsx")
```

---

## 18. Apéndice B — Glosario corto EN/ES

| EN | ES / nota |
| --- | --- |
| DataFrame | tabla / marco de datos |
| Series | serie / columna etiquetada |
| dtype | tipo de dato de columna |
| missing / NA | valor ausente / nulo |
| filter / mask | filtro / máscara booleana |
| groupby | agrupación |
| aggregate (`agg`) | agregar / resumir |
| vectorized | vectorizado |
| feature / target | variable predictora / objetivo |
| dashboard | cuadro de mando |

---

## 19. Apéndice C — Preguntas típicas de clase

**¿Puedo hacer la limpieza solo en Excel y pegar el CSV?**  
Para aprender, no: el valor está en el **código reproducible** que luego usará Dash.

**¿Obligatorio usar `pathlib`?**  
Sí en el material del curso: evita rutas de tu portátil.

**¿Por qué falló `read_excel`?**  
Casi seguro falta `openpyxl` (o el motor que uses) en el entorno.

**¿CSV, JSON y Excel deben coincidir siempre?**  
En este material de iris, sí. En la vida real, documenta el formato canónico.

**¿Esto sustituye al notebook de modelos?**  
No. Aquí preparas datos; en `6_modelos_machine_learning.ipynb` entrenas.

---

## 20. Cierre

Si dominas este notebook, tienes la base de datos de DAVD:

> **Leer con rutas portables → diagnosticar → transformar → exportar, listo para modelo o dashboard.**

**Siguiente paso inmediato:** [`5_ejercicios_pandas.md`](5_ejercicios_pandas.md) (ventas) y, si puedes, el E1 de airports en [`7_ejercicios_datos_y_modelos.md`](7_ejercicios_datos_y_modelos.md).


---

## 21. Ejercicio guiado E1 — `ventas.csv` (30 min)

Enunciado completo: [`5_ejercicios_pandas.md`](5_ejercicios_pandas.md).

Objetivo: dejar datos listos para un **cuadro de mando** (válidos vs errores + KPIs + JSON de calidad).

- **Dataset A (`ventas.csv`)** — checkpoint **140 / 10**
- **Dataset B (`penguins.csv`, abierto)** — checkpoint **333 / 11**

### 21.1 Carga y diagnóstico


In [37]:
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path("Datos")
ventas = pd.read_csv(DATA_DIR / "ventas.csv")

print("shape:", ventas.shape)
print(ventas.dtypes)
print(ventas.isna().sum())
ventas.head()


shape: (150, 6)
fecha               object
region              object
producto            object
unidades            object
precio_unitario    float64
cliente_id          object
dtype: object
fecha              0
region             0
producto           0
unidades           2
precio_unitario    1
cliente_id         0
dtype: int64


,fecha,region,producto,unidades,precio_unitario,cliente_id
0,2026-01-09,Sur,Cable Kit,7,12.50,C005
1,2026-03-14,Oeste,Gateway,25,120.00,C032
2,2026-01-26,Oeste,Hub IoT,1,88.96,C016
3,2026-02-05,Norte,Gateway,0,120.00,C010
4,2026-02-07,Norte,Hub IoT,15,85.00,C027


### 21.2 Validación

Implementa `validar_ventas` y comprueba el checkpoint.


In [38]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    work = frame.copy()
    work["unidades"] = pd.to_numeric(work["unidades"], errors="coerce")
    work["precio_unitario"] = pd.to_numeric(work["precio_unitario"], errors="coerce")
    ok = (
        work["unidades"].notna()
        & (work["unidades"] > 0)
        & work["precio_unitario"].notna()
        & (work["precio_unitario"] > 0)
    )
    validos = work.loc[ok].copy()
    errores = work.loc[~ok].copy()
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores


validos, errores = validar_ventas(ventas)
print(f"válidas: {len(validos)} | inválidas: {len(errores)}")
errores


válidas: 140 | inválidas: 10


,fecha,region,producto,unidades,precio_unitario,cliente_id
3,2026-02-05,Norte,Gateway,0.0,120.0,C010
41,2026-02-09,Oeste,Cable Kit,4.0,0.0,C015
54,2026-02-01,Sur,Sensor B,NaN,32.0,C004
63,2026-02-11,Norte,Sensor A,NaN,19.5,C018
66,2026-02-17,Oeste,Hub IoT,5.0,NaN,C025
84,2026-02-15,Este,Gateway,NaN,120.0,C022
88,2026-02-07,Este,Hub IoT,-2.0,85.0,C012
109,2026-02-19,Norte,Cable Kit,0.0,-1.0,C030
134,2026-02-03,Sur,Sensor A,3.0,-5.0,C007
149,2026-02-13,Sur,Sensor B,8.0,-32.0,C020


### 21.3 Agregaciones (KPIs)


In [39]:
importe_por_region = (
    validos.groupby("region", as_index=False)["importe"]
    .sum()
    .sort_values("importe", ascending=False)
)
top_productos = (
    validos.groupby("producto", as_index=False)["importe"]
    .sum()
    .sort_values("importe", ascending=False)
    .head(3)
)
compras_por_cliente = validos["cliente_id"].value_counts()
clientes_recurrentes = compras_por_cliente[compras_por_cliente > 1]

print("Importe por región:")
print(importe_por_region)
print("\nTop productos:")
print(top_productos)
print("\nClientes recurrentes:")
print(clientes_recurrentes)


Importe por región:
  region   importe
0   Este  33122.53
1  Norte  22914.63
3    Sur  21396.35
2  Oeste  19164.66

Top productos:
   producto   importe
1   Gateway  40440.00
2   Hub IoT  33058.66
4  Sensor B  11568.66

Clientes recurrentes:
C027    9
C032    8
C004    8
C035    7
C009    6
C013    6
C005    5
C022    5
C038    5
C034    4
C023    4
C024    4
C015    4
C025    4
C040    4
C002    4
C018    4
C010    4
C037    4
C011    3
C029    3
C030    3
C007    3
C003    3
C016    3
C033    3
C017    2
C020    2
C001    2
C021    2
C031    2
C039    2
C036    2
C028    2
Name: cliente_id, dtype: int64


### 21.4 Exportación


In [40]:
validos.to_csv(DATA_DIR / "ventas_limpias.csv", index=False)

calidad = {
    "filas_totales": int(len(ventas)),
    "filas_validas": int(len(validos)),
    "filas_invalidas": int(len(errores)),
    "importe_total": float(validos["importe"].sum()),
}
(DATA_DIR / "calidad_ventas.json").write_text(
    json.dumps(calidad, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
calidad


{'filas_totales': 150,
 'filas_validas': 140,
 'filas_invalidas': 10,
 'importe_total': 96598.17000000001}

Si el checkpoint no cuadra, revisa las reglas `> 0` y `to_numeric(..., errors="coerce")`. Detalle del enunciado en [`5_ejercicios_pandas.md`](5_ejercicios_pandas.md).


### 21.5 Dataset abierto — `penguins.csv`

Misma estructura: diagnóstico → validación → KPIs → export.  
Fuente: Palmer Penguins (ver [`Datos/README.md`](Datos/README.md)).

**Checkpoint B:** **333 válidas** y **11 inválidas**.


In [41]:
penguins = pd.read_csv(DATA_DIR / "penguins.csv")
print("shape:", penguins.shape)
print(penguins.isna().sum())
penguins.head()


shape: (344, 7)
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [42]:
def validar_penguins(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    work = frame.copy()
    for col in ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]:
        work[col] = pd.to_numeric(work[col], errors="coerce")
    work["sex"] = work["sex"].astype("string").str.upper()
    ok = (
        work["bill_length_mm"].notna() & (work["bill_length_mm"] > 0)
        & work["bill_depth_mm"].notna() & (work["bill_depth_mm"] > 0)
        & work["flipper_length_mm"].notna() & (work["flipper_length_mm"] > 0)
        & work["body_mass_g"].notna() & (work["body_mass_g"] > 0)
        & work["sex"].isin(["MALE", "FEMALE"])
        & work["species"].notna()
        & work["island"].notna()
    )
    return work.loc[ok].copy(), work.loc[~ok].copy()


penguins_ok, penguins_bad = validar_penguins(penguins)
print(f"válidas: {len(penguins_ok)} | inválidas: {len(penguins_bad)}")
penguins_bad.head()


válidas: 333 | inválidas: 11


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,<NA>
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,<NA>
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,<NA>
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,<NA>
11,Adelie,Torgersen,37.8,17.3,180.0,3700.0,<NA>


In [43]:
masa_por_especie = (
    penguins_ok.groupby("species", as_index=False)["body_mass_g"].mean()
    .sort_values("body_mass_g", ascending=False)
)
conteo_isla_sex = (
    penguins_ok.groupby(["island", "sex"], as_index=False).size()
    .rename(columns={"size": "n"})
)
top_flipper = penguins_ok.nlargest(5, "flipper_length_mm")[
    ["species", "island", "sex", "flipper_length_mm", "body_mass_g"]
]

print(masa_por_especie)
print(conteo_isla_sex.head())
print(top_flipper)

penguins_ok.to_csv(DATA_DIR / "penguins_limpios.csv", index=False)
calidad_p = {
    "filas_totales": int(len(penguins)),
    "filas_validas": int(len(penguins_ok)),
    "filas_invalidas": int(len(penguins_bad)),
    "body_mass_mean": float(penguins_ok["body_mass_g"].mean()),
}
(DATA_DIR / "calidad_penguins.json").write_text(
    json.dumps(calidad_p, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
calidad_p


     species  body_mass_g
2     Gentoo  5092.436975
1  Chinstrap  3733.088235
0     Adelie  3706.164384
      island     sex   n
0     Biscoe  FEMALE  80
1     Biscoe    MALE  83
2      Dream  FEMALE  61
3      Dream    MALE  62
4  Torgersen  FEMALE  24
    species  island   sex  flipper_length_mm  body_mass_g
283  Gentoo  Biscoe  MALE              231.0       5650.0
221  Gentoo  Biscoe  MALE              230.0       5700.0
253  Gentoo  Biscoe  MALE              230.0       6050.0
285  Gentoo  Biscoe  MALE              230.0       5700.0
295  Gentoo  Biscoe  MALE              230.0       5800.0


{'filas_totales': 344,
 'filas_validas': 333,
 'filas_invalidas': 11,
 'body_mass_mean': 4207.057057057057}